In [ ]:
import astropy.units as u
import numpy as np
import xarray as xr
from astropy.coordinates import SkyCoord

from astropy_xarray.coordinates.sky_coord import (
    skycoord_to_dataset,
)


# zarr
def to_zarr(ds: xr.Dataset):
    import zarr.storage as zs

    store = zs.MemoryStore()
    return store, ds.to_zarr(store, mode="w")


# zarr.tar.gz
def to_zarr_tar_gz(ds: xr.Dataset):
    import io
    import tarfile

    store, obj = to_zarr(ds)
    with store:
        buf = io.BytesIO()
        with tarfile.open(fileobj=buf, mode="w:gz", compresslevel=9) as tar:
            for key, value in store._store_dict.items():
                info = tarfile.TarInfo(name=key)
                info.size = len(value)
                tar.addfile(tarinfo=info, fileobj=io.BytesIO(value.as_buffer_like()))
    buf.seek(0)
    return buf.getbuffer()


def from_zarr_tar_gz(buf: memoryview):
    import io
    import tarfile

    import zarr.storage as zs
    from zarr.core.buffer.cpu import Buffer

    store = zs.MemoryStore()

    # Open tar.gz from buffer
    with tarfile.open(fileobj=io.BytesIO(buf), mode="r:gz") as tar:
        for member in tar.getmembers():
            f = tar.extractfile(member)
            # Read raw bytes into the Zarr store
            store._store_dict[member.name] = Buffer.from_bytes(f.read())
    return xr.open_zarr(store)


# zarr.zip
def to_zarr_zip(ds: xr.Dataset):
    import io
    import zipfile

    store, obj = to_zarr(ds)
    with store:
        buf = io.BytesIO()
        with zipfile.ZipFile(buf, mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
            for key, value in store._store_dict.items():
                zf.writestr(key, value.as_buffer_like())
        buf.seek(0)
    return buf.getbuffer()


def from_zarr_zip(buf: memoryview):
    import io
    import zipfile

    import zarr.storage as zs
    from zarr.core.buffer.cpu import Buffer

    store = zs.MemoryStore()

    # Open tar.gz from buffer
    with zipfile.ZipFile(
        io.BytesIO(buf), mode="r", compression=zipfile.ZIP_DEFLATED
    ) as zf:
        for name in zf.namelist():
            # Read raw bytes into the Zarr store
            store._store_dict[name] = Buffer.from_bytes(zf.read(name))
    return xr.open_zarr(store)


# msgpack_numpy
def to_msgpack_numpy(ds: xr.Dataset):
    import msgpack_numpy

    return msgpack_numpy.packb(ds.to_dict(data="array"))


def from_msgpack_numpy(buf: memoryview):
    import msgpack_numpy

    return xr.Dataset.from_dict(msgpack_numpy.unpackb(buf))


# msgpack
def to_msgpack(ds: xr.Dataset):
    import msgpack

    return msgpack.packb(ds.to_dict())


def from_msgpack(buf: memoryview):
    import msgpack

    return xr.Dataset.from_dict(msgpack.unpackb(buf))


# json
def to_json(ds: xr.Dataset):
    import json

    return json.dumps(ds.to_dict(), separators=(",", ":"))


def from_json(buf: str):
    import json

    return xr.Dataset.from_dict(json.loads(buf))


def gen_skycoord_ds(count):
    sc = SkyCoord(
        ra=np.random.random(count) * u.rad, dec=np.random.random(count) * u.rad
    )
    return skycoord_to_dataset(sc).astropy.dequantify()

In [ ]:
import ipytest

ipytest.autoconfig()

In [ ]:
%%ipytest -qq

from pytest_benchmark.fixture import BenchmarkFixture

count = 10000
sc = gen_skycoord_ds(count)

# Storage for results
bench_data = {}

def test_to_zarr_tar_gz(benchmark: BenchmarkFixture):
    result = benchmark(to_zarr_tar_gz, sc)
    assert from_zarr_tar_gz(result) == sc
    bench_data['to_zarr_tar_gz'] = benchmark.stats

def test_from_zarr_tar_gz(benchmark: BenchmarkFixture):
    raw = to_zarr_tar_gz(sc)
    result = benchmark(from_zarr_tar_gz, raw)
    assert result == sc
    bench_data['from_zarr_tar_gz'] = benchmark.stats

def test_to_zarr_zip(benchmark: BenchmarkFixture):
    result = benchmark(to_zarr_zip, sc)
    bench_data['to_zarr_zip'] = benchmark.stats
    assert len(result)

def test_from_zarr_zip(benchmark: BenchmarkFixture):
    raw = to_zarr_zip(sc)
    result = benchmark(from_zarr_zip, raw)
    assert result == sc
    bench_data['from_zarr_zip'] = benchmark.stats


def test_to_msgpack_numpy(benchmark: BenchmarkFixture):
    result = benchmark(to_msgpack_numpy, sc)
    bench_data['to_msgpack_numpy'] = benchmark.stats
    assert len(result)

def test_from_msgpack_numpy(benchmark: BenchmarkFixture):
    raw = to_msgpack_numpy(sc)
    result = benchmark(from_msgpack_numpy, raw)
    assert result == sc
    bench_data['from_msgpack_numpy'] = benchmark.stats

def test_to_msgpack(benchmark: BenchmarkFixture):
    result = benchmark(to_msgpack, sc)
    bench_data['to_msgpack'] = benchmark.stats
    assert len(result)

def test_from_msgpack(benchmark: BenchmarkFixture):
    raw = to_msgpack(sc)
    result = benchmark(from_msgpack, raw)
    assert result == sc
    bench_data['from_msgpack'] = benchmark.stats

def test_to_json(benchmark: BenchmarkFixture):
    result = benchmark(to_json, sc)
    bench_data['to_json'] = benchmark.stats
    assert len(result)

def test_from_json(benchmark: BenchmarkFixture):
    raw = to_json(sc)
    result = benchmark(from_json, raw)
    assert result == sc
    bench_data['from_json'] = benchmark.stats

In [ ]:
records = []
for name, meta in bench_data.items():
    record = {
        "name": name,
        "mean": meta.stats.mean * u.s,
        "stddev": meta.stats.stddev,
        "min": meta.stats.min * u.s,
        "max": meta.stats.max * u.s,
        "median": meta.stats.median * u.s,
        "rounds": meta.stats.rounds,
        "iqr": meta.stats.iqr * u.s,
        "outliers": meta.stats.outliers,
    }
    records.append(record)

records
# dataset = xr.Dataset.from_dataframe(pd.DataFrame(records))
# dataset

In [ ]:
import pandas as pd
from pytest_benchmark.stats import Metadata


def create_perf_stats_dataset(bench_data: dict[str, Metadata]):
    records = []
    for name, meta in bench_data.items():
        record = {
            "name": name,
            "mean": meta.stats.mean,  # * u.s,
            "stddev": meta.stats.stddev,
            "min": meta.stats.min,  # * u.s,
            "max": meta.stats.max,  # * u.s,
            "median": meta.stats.median,  # * u.s,
            "rounds": meta.stats.rounds,
            "iqr": meta.stats.iqr,  # * u.s,
            "q1": meta.stats.q1,  # * u.s,
            "q3": meta.stats.q3,  # * u.s,
            "outliers": meta.stats.outliers,
        }
        records.append(record)

    dataset = xr.Dataset.from_dataframe(pd.DataFrame(records))
    dataset = (
        xr.concat(
            [
                dataset.loc[{"index": slice(0, None, 2)}]
                .rename_dims({"index": "format"})
                .drop_vars("index"),
                dataset.loc[{"index": slice(1, None, 2)}]
                .rename_dims({"index": "format"})
                .drop_vars("index"),
            ],
            dim="readwrite",
        )
        .assign_coords(readwrite=["read", "write"])
        .rename_vars({"name": "func_name"})
        .expand_dims("sample")
        .assign_coords(
            format_desc=(
                "format",
                [
                    "Zarr as Gzipped Tape Archive",
                    "Zarr as Zip Archive",
                    "MessagePack Numpy",
                    "MessagePack",
                    "JSON",
                ],
            ),
            format_ext=(
                "format",
                [".zarr.tar.gz", ".zarr.zip", ".mpknp", ".mpk", ".json"],
            ),
            sample_len=("sample", [10000]),
        )
    )
    return dataset


perf_stats = create_perf_stats_dataset(bench_data)
display(perf_stats)

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

elements = [10, 100, 1000, 10000]
ser_data = [
    [to_zarr_tar_gz(gen_skycoord_ds(count)) for count in elements],
    [to_zarr_zip(gen_skycoord_ds(count)) for count in elements],
    [to_msgpack_numpy(gen_skycoord_ds(count)) for count in elements],
    [to_msgpack(gen_skycoord_ds(count)) for count in elements],
    [to_json(gen_skycoord_ds(count)) for count in elements],
]

In [ ]:
compression_info = xr.Dataset(
    coords=dict(
        format_desc=(
            "format",
            [
                "Zarr as Gzipped Tape Archive",
                "Zarr as Zip Archive",
                "MessagePack Numpy",
                "MessagePack",
                "JSON",
            ],
        ),
        format_ext=("format", [".zarr.tar.gz", ".zarr.zip", ".mpknp", ".mpk", ".json"]),
        sample_len=("sample", elements),
    ),
    data_vars=dict(
        size=(
            ["sample", "format"],
            np.array([[len(sample) for sample in row] for row in ser_data]).T,
        ),
        compression_ratio=(
            ["format"],
            np.array([16 * (elements[-1]) / (len(row[-1])) for row in ser_data]),
        ),
    ),
)
display(compression_info)

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(1, (20, 16))
axes = fig.subplots(2, 2)
ax0 = axes[0, 0]
lines = compression_info.size.plot.line(
    x="sample_len",
    hue="format_desc",
    marker="v",
    xscale="log",
    yscale="log",
    ax=ax0,
)
ax0.set_title("serialization compression size")

ax1 = axes[0, 1]
ax2 = axes[1, 0]
ax3 = axes[1, 1]

# uncompressed / compressed (200,000 float64)
compression_info.compression_ratio.to_dataframe().plot.bar(
    title="serialization compression-ratio (10000)",
    x="format_ext",
    rot=20,
    ax=ax1,
    legend=False,
)
for bars in ax1.containers:
    ax1.bar_label(bars)

boxplot_style = dict(
    patch_artist=False,
    showmeans=True,
    meanline=True,
    showfliers=False,
    shownotches=True,
)

write_perf = (
    perf_stats[["min", "max", "median", "mean", "q1", "q3", "iqr", "rounds"]]
    .loc[{"readwrite": "write"}]
    .to_dataframe()
)
ax2.bxp(
    [
        dict(
            med=row["median"],
            cilo=row["median"] - (1.57 * row.iqr / np.sqrt(row["rounds"])),
            cihi=row["median"] + (1.57 * row.iqr / np.sqrt(row["rounds"])),
            q1=row["q1"],
            q3=row["q3"],
            whislo=row["min"],
            whishi=row["max"],
            label=f"{row.format_ext}\nn={row['rounds']}",
            mean=row["mean"],
            fliers=0,
        )
        for row in write_perf.iloc
    ],
    **boxplot_style,
)
ax2.set_title("in-memory write performance (10000)")
ax2.set_ylim(0, 0.03)


read_perf = (
    perf_stats[["min", "max", "median", "mean", "q1", "q3", "iqr", "rounds"]]
    .loc[{"readwrite": "read"}]
    .to_dataframe()
)
ax3.set_title("in-memory read performance (10000)")
ax3.set_ylim(0, 0.03)
read_plot = ax3.bxp(
    [
        dict(
            med=row["median"],
            cilo=row["median"] - (1.57 * row.iqr / np.sqrt(row["rounds"])),
            cihi=row["median"] + (1.57 * row.iqr / np.sqrt(row["rounds"])),
            q1=row["q1"],
            q3=row["q3"],
            whislo=row["min"],
            whishi=row["max"],
            label=f"{row.format_ext}\nn={row['rounds']}",
            mean=row["mean"],
            fliers=0,
        )
        for row in read_perf.iloc
    ],
    **boxplot_style,
)

## Conclusions

* .zarr.tar.gz: Max compression
* .zarr.zip: Random data_var without extract
* .mpknp: Fast in-memory serialization
* .json: Human readable payloads